In [ ]:
import pandas as pd
import networkx as nx
from pathlib import Path

# ===============================
# 1️⃣ 构建加权三层网络
# ===============================
def build_weighted_multilayer_network(df, zone_col='zone', host_col='host', virus_col='virus'):
    G = nx.Graph()

    # zone-host 权重
    zh_weight = df.groupby([zone_col, host_col]).size().to_dict()
    # host-virus 权重
    hv_weight = df.groupby([host_col, virus_col]).size().to_dict()

    zones = df[zone_col].unique()
    hosts = df[host_col].unique()
    viruses = df[virus_col].unique()

    G.add_nodes_from(zones, layer='zone')
    G.add_nodes_from(hosts, layer='host')
    G.add_nodes_from(viruses, layer='virus')

    for (z, h), w in zh_weight.items():
        G.add_edge(z, h, weight=w)
    for (h, v), w in hv_weight.items():
        G.add_edge(h, v, weight=w)

    return G

# ===============================
# 2️⃣ 构建 Zone–Zone 最低阻力网络
# ===============================
def build_zone_flow_network(G):
    zones = [n for n, d in G.nodes(data=True) if d['layer'] == 'zone']

    G_zone = nx.Graph()  # 无向网络

    for i in range(len(zones)):
        for j in range(i + 1, len(zones)):
            zi, zj = zones[i], zones[j]
            flow = 0.0
            for h in G.neighbors(zi):
                if G.nodes[h]['layer'] != 'host':
                    continue
                w_zh = G[zi][h]['weight']
                for v in G.neighbors(h):
                    if G.nodes[v]['layer'] != 'virus':
                        continue
                    w_hv = G[h][v]['weight']
                    for h2 in G.neighbors(v):
                        if G.nodes[h2]['layer'] != 'host':
                            continue
                        if not G.has_edge(h2, zj):
                            continue
                        w_vh2 = G[v][h2]['weight']
                        w_h2z = G[h2][zj]['weight']
                        flow += w_zh * w_hv * w_vh2 * w_h2z
            if flow > 0:
                G_zone.add_edge(
                    zi, zj,
                    flow=flow,
                    resistance=1.0 / (flow + 1e-6)
                )
    return G_zone

# ===============================
# 3️⃣ 计算最低阻力路径（优化）
# ===============================
def find_least_resistance_paths(G_zone, top_n=10):
    paths = []
    zones = list(G_zone.nodes)

    for i in range(len(zones)):
        for j in range(i + 1, len(zones)):
            zi, zj = zones[i], zones[j]
            try:
                length = nx.shortest_path_length(G_zone, zi, zj, weight='resistance')
                path = nx.shortest_path(G_zone, zi, zj, weight='resistance')

                # 无向网络强制 source < target
                source, target = sorted([zi, zj])

                # 如果 path 第一个节点不是 source，则反转 path
                if path[0] != source:
                    path = path[::-1]

                paths.append({
                    'source_zone': source,
                    'target_zone': target,
                    'resistance': length,
                    'path': ' → '.join(map(str, path)),
                    'flow': sum(G_zone[path[k]][path[k+1]]['flow'] for k in range(len(path)-1))
                })
            except nx.NetworkXNoPath:
                continue

    return pd.DataFrame(paths).sort_values('resistance').head(top_n)

# ===============================
# 4️⃣ 主程序
# ===============================
if __name__ == "__main__":
    # 读取数据
    df = pd.read_excel(r'input/host_virus-NEW-0313.xlsx')
    df = df[df['virus_new'].notna()].reset_index(drop=True)
    df = df[['zone', 'Host_species', 'virus_new', 'County']]
    df = df.rename(columns={'Host_species': 'host', 'virus_new': 'virus'})

    # 拆分多病毒行
    df_new = (
        df
        .assign(virus=df['virus'].str.split(','))
        .explode('virus')
        .assign(virus=lambda x: x['virus'].str.strip())
        .reset_index(drop=True)
    )
    df_new[['host', 'virus']] = df_new[['host', 'virus']].apply(lambda x: x.str.strip())
    

    # 构建加权三层网络
    G = build_weighted_multilayer_network(df_new)

    # 构建 zone–zone 流网络
    G_zone = build_zone_flow_network(G)

    # 计算最低阻力路径
    least_paths = find_least_resistance_paths(G_zone, top_n=10)

    # 输出
    outdir = Path('output/1_flow_analysis')
    outdir.mkdir(parents=True, exist_ok=True)
    least_paths.to_excel(outdir / 'least_resistance_paths.xlsx', index=False)

    print("最低阻力路径计算完成，结果输出到：", outdir / 'least_resistance_paths.xlsx')

最低阻力路径计算完成，结果输出到： output\1_flow_analysis\least_resistance_paths.xlsx
